# 05 ? ??nh gi? V3 EMA ? A1 tr?n Table 2, 3 v? 4

G?n dataset ch?a 10 checkpoint v? manifest, s?a `CHECKPOINT_FOLDER`, b?t GPU v? Internet. M?i seed gh?p checkpoint EMA v? A1 c?ng seed. ?i?m l? t?ch x?c su?t human, ng??ng 0,5. M?i t?p t?o m?t graph V3, test ?? 5 seed r?i x?a graph tr??c khi chuy?n sang t?p ti?p theo.


In [ ]:
%pip install -q torch-geometric transformers==4.44.2 'spacy>=3.8,<3.9' scikit-learn tqdm
!python -m spacy download en_core_web_sm


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, hashlib, shutil

BRANCH = 'version3'
REPO_URL = 'https://github.com/tphan1420/PRDetect-Research.git'
WORK = Path('/kaggle/working/v3_ema_a1_tables')
REPO = WORK / 'PRDetect-Research'
# Upload ZIP as a Kaggle Dataset, extract its contents, and set this folder.
CHECKPOINT_FOLDER = Path('/kaggle/input/datasets/your-account/v3-ema-a1-checkpoints')
SEEDS = [2020, 2022, 2024, 2025, 2026]
BATCH_SIZE = 4
WORK.mkdir(parents=True, exist_ok=True)
assert CHECKPOINT_FOLDER.is_dir(), f'S?a CHECKPOINT_FOLDER: {CHECKPOINT_FOLDER}'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
manifest_files = list(CHECKPOINT_FOLDER.rglob('manifest.json'))
assert len(manifest_files) == 1, 'C?n manifest.json trong th? m?c checkpoint'
manifest = json.loads(manifest_files[0].read_text(encoding='utf8'))
assert isinstance(manifest, list)
checkpoint_dir = manifest_files[0].parent
for record in manifest:
    path = checkpoint_dir / record['file']
    assert path.is_file(), path
    assert hashlib.sha256(path.read_bytes()).hexdigest() == record['sha256'], path
for seed in SEEDS:
    for filename in (f'v3_baseline_ema_seed{seed}.pt', f'a1_seed{seed}.pt'):
        assert sum(r['file'] == filename and r['seed'] == seed for r in manifest) == 1, filename
print('?? ki?m tra 5 c?p checkpoint')


In [ ]:
%%writefile v3_a1_fusion.py
"""Score text with the HC3 A1 checkpoint and combine it with V3 predictions."""

import argparse
import hashlib
import json
from pathlib import Path

import numpy as np
from scipy.special import expit

from evaluate_v3_quang import REPO, score_probabilities, sha256, write_json


def source_digest(path):
    # Kaggle's LF checkout and Windows' CRLF checkout contain the same records.
    return hashlib.sha256(path.read_bytes().replace(b"\r\n", b"\n")).hexdigest()


def human_score(v3_p_human, a1_logit_human):
    """A soft AND. The positive class (human) requires support from both views."""
    p = np.asarray(v3_p_human, dtype=np.float64)
    z = np.asarray(a1_logit_human, dtype=np.float64)
    if (p.shape != z.shape or not np.isfinite(p).all() or not np.isfinite(z).all()
            or np.any((p < 0) | (p > 1))):
        raise ValueError("Expected aligned finite V3 probabilities and A1 human logits")
    return p * expit(z)


def read_source(path):
    with path.open(encoding="utf-8") as stream:
        return [json.loads(line) for line in stream if line.strip()]


def labels_from(rows):
    labels = []
    for row in rows:
        value = row["label"]
        if isinstance(value, str):
            if "human" in value.lower():
                value = 1
            elif any(name in value.lower() for name in ("gpt", "ai", "machine")):
                value = 0
        if value not in (0, 1):
            raise ValueError(f"Unknown binary label {value!r}")
        labels.append(int(value))
    return np.asarray(labels)


def paired_scores(source, labels, v3, a1):
    if (v3["split"] != source.stem or a1["condition"] != source.stem
            or a1["source_sha256"] not in {source_digest(source), sha256(source)}
            or a1["score_space"] != "human_logit" or v3["train"] != "hc3"
            or v3["label_convention"] != {"0": "ai", "1": "human"}):
        raise ValueError(f"Score provenance differs from source {source.name}")
    records = a1["records"]
    if (v3["samples"] != len(labels) or len(records) != len(labels)
            or [r["index"] for r in records] != list(range(len(labels)))
            or not np.array_equal(labels, v3["y_true"])
            or not np.array_equal(labels, [r["label"] for r in records])):
        raise ValueError(f"Sample order or labels differ in {source.name}")
    return human_score(v3["p_human"], [r["logit_human"] for r in records])


class AlignedScorer:
    """The original A1 aligned RoBERTa and V1 dependency graph, streamed by text."""

    def __init__(self, checkpoint, device):
        import spacy
        import torch
        from transformers import AutoModel, AutoTokenizer
        from dep_vocab import get_default_vocab

        self.device = torch.device(device)
        self.nlp = spacy.load("en_core_web_sm", disable=["ner"])
        self.vocab = get_default_vocab()
        self.tokenizer = AutoTokenizer.from_pretrained("roberta-base", use_fast=True, add_prefix_space=True)
        self.encoder = AutoModel.from_pretrained("roberta-base", add_pooling_layer=False).to(device).eval()
        if checkpoint is None:
            return
        from model.RGCN import RGCN2
        state = torch.load(checkpoint, weights_only=True, map_location="cpu")
        relations, bases = state["conv1.comp"].shape
        self.graph = RGCN2(input_dim=state["conv1.root"].shape[0], hidden_dim=state["conv1.root"].shape[1],
                           output_dim=state["fc.weight"].shape[1], num_relations=relations, num_bases=bases)
        self.graph.load_state_dict(state, strict=True)
        self.graph.to(device).eval()
        # At inference the same basis weights are expanded for every document.
        # Materialize them once with the identical PyG basis product.
        with torch.no_grad():
            for layer in (self.graph.conv1, self.graph.conv2):
                basis = layer.weight
                dense = (layer.comp @ basis.flatten(1)).view(relations, *basis.shape[1:])
                layer.weight = torch.nn.Parameter(dense, requires_grad=False)
                layer.num_bases = None

    def score_doc(self, doc):
        import torch
        if not len(doc):
            raise ValueError("Cannot classify an empty document")
        tokens = self.tokenizer([w.text for w in doc], is_split_into_words=True, max_length=512,
                                stride=64, truncation=True, return_overflowing_tokens=True,
                                padding=True, return_tensors="pt")
        with torch.inference_mode():
            sums = torch.zeros((len(doc), self.encoder.config.hidden_size), device=self.device)
            counts = torch.zeros(len(doc), device=self.device)
            for i in range(len(tokens["input_ids"])):
                # Padding contributes neither to attention nor to word pooling.
                size = int(tokens["attention_mask"][i].sum())
                ids = tokens["input_ids"][i:i + 1, :size].to(self.device)
                mask = tokens["attention_mask"][i:i + 1, :size].to(self.device)
                with torch.autocast("cuda", enabled=self.device.type == "cuda"):
                    hidden = self.encoder(input_ids=ids, attention_mask=mask).last_hidden_state[0]
                word_ids = tokens.word_ids(i)[:size]
                positions = [j for j, word in enumerate(word_ids) if word is not None]
                words = torch.tensor([word_ids[j] for j in positions], device=self.device)
                sums.index_add_(0, words, hidden[positions].float())
                counts.index_add_(0, words, torch.ones_like(words, dtype=sums.dtype))
            x = sums / counts.clamp_min(1).unsqueeze(1)
            edges = torch.tensor([[w.i for w in doc], [w.head.i for w in doc]], device=self.device)
            rel = torch.tensor([self.vocab.encode(w.dep_) for w in doc], device=self.device)
            h = self.graph.conv1(x, edges, rel).relu()
            h = self.graph.conv2(h, edges, rel).relu()
            return self.graph.fc(h).mean().item()

    def v3_probability(self, doc, model):
        """Use the V3 checkpoint's original word IDs, edge schema and FP16 cache rounding."""
        import torch
        from model.QuangV3 import struct_features
        words = [w.text for w in doc]
        if not words:
            raise ValueError("Cannot classify an empty document")
        src = np.arange(len(doc))
        dst = np.array([w.head.i for w in doc])
        rel = np.array([self.vocab.encode(w.dep_) for w in doc]).clip(max=model.R - 1)
        structure, _ = struct_features(len(doc), src, dst, rel, model.R, model.mode)
        with torch.inference_mode():
            chunks = []
            for start in range(0, len(words), 512):
                ids = self.tokenizer.convert_tokens_to_ids(words[start:start + 512])
                inputs = torch.tensor([ids], device=self.device)
                chunks.append(self.encoder(inputs).last_hidden_state[0].half().float())
            batch = {"x": torch.cat(chunks), "src": torch.as_tensor(src, device=self.device),
                     "dst": torch.as_tensor(dst, device=self.device), "rel": torch.as_tensor(rel, device=self.device),
                     "st": torch.as_tensor(structure, dtype=torch.long, device=self.device),
                     "B": 1, "bidx": torch.zeros(len(words), dtype=torch.long, device=self.device)}
            return model(batch).sigmoid().item()


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    commands = parser.add_subparsers(dest="command", required=True)
    score = commands.add_parser("score-a1")
    score.add_argument("--sources", type=Path, nargs="+", required=True)
    score.add_argument("--output-dir", type=Path, required=True)
    score.add_argument("--checkpoint", type=Path, default=REPO / "model/hc3_a1_only_rgcn_model_2026.pth")
    score.add_argument("--device", choices=("cpu", "cuda"), default="cpu")
    predict = commands.add_parser("predict", help="Classify a new English document with a single V3 seed and A1")
    predict.add_argument("--text", required=True)
    predict.add_argument("--v3-checkpoint", required=True, type=Path)
    predict.add_argument("--rule", choices=("fusion", "human95", "routed"), default="fusion")
    predict.add_argument("--router", type=Path, help="Fitted hc3_router.joblib for the routed rule")
    predict.add_argument("--human95-dir", type=Path, help="Source thresholds for checkpoints created before human95")
    predict.add_argument("--checkpoint", type=Path, default=REPO / "model/hc3_a1_only_rgcn_model_2026.pth")
    predict.add_argument("--device", choices=("cpu", "cuda"), default="cpu")
    fuse = commands.add_parser("fuse")
    for name in ("source", "a1", "v3", "output"):
        fuse.add_argument("--" + name, type=Path, required=True)
    args = parser.parse_args()
    if args.command == "fuse":
        labels = labels_from(read_source(args.source))
        v3, a1 = [json.loads(p.read_bytes()) for p in (args.v3, args.a1)]
        p = paired_scores(args.source, labels, v3, a1)
        write_json(args.output, {"split": args.source.stem, "threshold": 0.5, "seed": v3["seed"],
                                 "a1_checkpoint": a1["checkpoint_sha256"], "v3_checkpoint": v3["checkpoint_sha256"],
                                 **score_probabilities(labels, p, 0.5), "human_score": p.tolist()})
        return
    if args.command == "predict":
        if args.rule == "routed":
            import joblib
            from v3_a1_routing import use_v3
            if args.router is None:
                parser.error("--rule routed requires --router")
            if not use_v3(joblib.load(args.router), [args.text])[0]:
                scorer = AlignedScorer(args.checkpoint, args.device)
                logit = scorer.score_doc(scorer.nlp(args.text))
                print(json.dumps({"rule": "routed", "branch": "a1", "human_score": float(expit(logit)),
                                  "threshold": 0.5, "a1_checkpoint_sha256": sha256(args.checkpoint),
                                  "label": "human" if logit >= 0 else "ai"}))
                return
        from calibrate_v3 import load_v3_checkpoint
        model, ckpt, key = load_v3_checkpoint(args.v3_checkpoint)
        if args.rule == "routed" and ckpt.get("weight_source") == "ema":
            raise ValueError("The routed method uses the V3 RAW checkpoint")
        scorer = AlignedScorer(args.checkpoint if args.rule == "fusion" else None, args.device)
        doc = scorer.nlp(args.text)
        v3 = scorer.v3_probability(doc, model.to(scorer.device))
        if args.rule in ("human95", "routed"):
            threshold = ckpt.get("human95_threshold")
            if args.human95_dir:
                fitted = json.loads((args.human95_dir / f"{key}.json").read_bytes())
                if fitted["checkpoint_sha256"] != key:
                    raise ValueError("Mismatched source calibration")
                threshold = fitted["human95_threshold"]
            if threshold is None or not 0 <= threshold <= 1:
                raise ValueError("Fit and supply --human95-dir for this checkpoint")
            result = {"human_score": v3, "threshold": threshold}
            if args.rule == "routed":
                result["branch"] = "v3_raw"
        else:
            a1 = scorer.score_doc(doc)
            result = {"human_score": float(human_score(v3, a1)), "threshold": 0.5, "a1_p_human": float(expit(a1))}
        result.update(rule=args.rule, v3_p_human=v3, v3_checkpoint_sha256=key,
                      label="human" if result["human_score"] >= result["threshold"] else "ai")
        print(json.dumps(result))
        return
    scorer = AlignedScorer(args.checkpoint, args.device)
    digest = sha256(args.checkpoint)
    cache = {}
    for source in args.sources:
        rows = read_source(source)
        texts = [r["text"] for r in rows]
        run = {"condition": source.stem, "source_sha256": source_digest(source),
               "checkpoint_sha256": digest, "score_space": "human_logit"}
        partial = args.output_dir / f"scores_{source.stem}.partial.json"
        signature = {**run, "scorer_sha256": sha256(Path(__file__)), "device": args.device}
        completed = {}
        if partial.exists():
            saved = json.loads(partial.read_bytes())
            if saved["signature"] != signature:
                raise ValueError(f"Partial scores used a different scorer or source: {partial}")
            completed = saved["logits"]
            for i, value in completed.items():
                cache[texts[int(i)]] = value
        missing = dict.fromkeys(t for t in texts if t not in cache)
        indices = {t: i for i, t in enumerate(texts)}
        for n, doc in enumerate(scorer.nlp.pipe(missing, batch_size=16), start=1):
            cache[doc.text] = scorer.score_doc(doc)
            completed[str(indices[doc.text])] = cache[doc.text]
            if n % 32 == 0:
                write_json(partial, {"signature": signature, "logits": completed})
                print(f"{source.stem}: {n}/{len(missing)} new texts", flush=True)
        records = [{"index": i, "label": row.get("label"), "logit_human": cache[texts[i]]}
                   for i, row in enumerate(rows)]
        if all(r["label"] is not None for r in records):
            for r, label in zip(records, labels_from(rows)):
                r["label"] = int(label)
        write_json(args.output_dir / f"scores_{source.stem}.json", {**run, "records": records})
        partial.unlink(missing_ok=True)
        print(f"{source.stem}: {len(rows)} documents", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
# (suite, condition, source JSON relative to REPO)
# Comment out a row to skip that test file.
CASES = [
    # Table 2: train HC3 -> test HC3 (in-domain lexical noise).
    ('table2', 'Original',     0, 'data/original_text/benchmarks/hc3/hc3_test.json'),
    ('table2', 'Synonym 5%',   5, 'data/perturbed_text/hc3/lexical_synonym/hc3_test_adj_5_synonym_replace.json'),
    ('table2', 'Synonym 10%', 10, 'data/perturbed_text/hc3/lexical_synonym/hc3_test_adj_10_synonym_replace.json'),
    ('table2', 'Synonym 20%', 20, 'data/perturbed_text/hc3/lexical_synonym/hc3_test_adj_20_synonym_replace.json'),
    ('table2', 'Synonym 30%', 30, 'data/perturbed_text/hc3/lexical_synonym/hc3_test_adj_30_synonym_replace.json'),

    # Table 3: train HC3 -> GPT-3.5 Mixed (cross-dataset lexical noise).
    ('table3', 'Original', 0, 'data/original_text/benchmarks/gpt35_mixed/gpt3.5_mixed_test_split.json'),
    ('table3', 'Synonym 5%',  5, 'data/perturbed_text/gpt35_mixed/lexical_synonym/gpt3.5_mixed_test_adj_5_synonym_replace.json'),
    ('table3', 'Synonym 10%', 10, 'data/perturbed_text/gpt35_mixed/lexical_synonym/gpt3.5_mixed_test_adj_10_synonym_replace.json'),
    ('table3', 'Synonym 20%', 20, 'data/perturbed_text/gpt35_mixed/lexical_synonym/gpt3.5_mixed_test_adj_20_synonym_replace.json'),
    ('table3', 'Synonym 30%', 30, 'data/perturbed_text/gpt35_mixed/lexical_synonym/gpt3.5_mixed_test_adj_30_synonym_replace.json'),

    # Table 4: HC3 structural noise.
    ('table4', 'Original',     0, 'data/original_text/benchmarks/hc3/hc3_test.json'),
    ('table4', 'Insert 25%',  25, 'data/perturbed_text/hc3/structural_noise/hc3_test_insert_25.json'),
    ('table4', 'Repeat 25%',  25, 'data/perturbed_text/hc3/structural_noise/hc3_test_repeat_25.json'),
    ('table4', 'Replace 25%', 25, 'data/perturbed_text/hc3/structural_noise/hc3_test_replace_25.json'),
    ('table4', 'Delete 25%',  25, 'data/perturbed_text/hc3/structural_noise/hc3_test_delete_25.json'),
]

for suite, condition, noise, source in CASES:
    assert (REPO / source).is_file(), source
print('S? ?i?u ki?n:', len(CASES), '| s? t?p duy nh?t:', len({c[3] for c in CASES}))


In [ ]:
import numpy as np
import pandas as pd
import torch
from types import SimpleNamespace
import evaluate_v3_quang as ev
from v3_a1_fusion import read_source, labels_from, paired_scores, source_digest

assert torch.cuda.is_available(), 'B?t GPU trong Kaggle Settings'
OUT = WORK / 'results'
OUT.mkdir(parents=True, exist_ok=True)
unique_sources = list(dict.fromkeys(source for _, _, _, source in CASES))
all_results = []
configs = {}
for seed in SEEDS:
    result_dir = OUT / f'seed{seed}'
    result_dir.mkdir(exist_ok=True)
    args = SimpleNamespace(variant='baseline', train_domain='hc3', checkpoint_dir=str(checkpoint_dir),
                           seeds=[seed], batch_size=BATCH_SIZE, struct_mode='low', device='cuda',
                           graph_budget_root=str(WORK), weights='ema')
    config = ev.make_config(args)
    v3_checkpoint = checkpoint_dir / f'v3_baseline_ema_seed{seed}.pt'
    assert Path(config['checkpoint_paths'][str(seed)]).resolve() == v3_checkpoint.resolve()
    config_path = result_dir / 'v3_config.json'
    ev.write_json(config_path, config)
    configs[seed] = config_path

for relative in unique_sources:
    source = REPO / relative
    graph = REPO / 'graph_data' / f'{source.stem}.pkl'
    matching = [(table, condition, noise) for table, condition, noise, path in CASES if path == relative]
    truth = labels_from(read_source(source))
    print('T?o graph V3', source.stem, flush=True)
    subprocess.run([sys.executable, 'building_graph.py', '--file', str(source)], cwd=REPO, check=True)
    assert graph.is_file(), graph
    try:
        for seed in SEEDS:
            print('??nh gi?', source.stem, 'seed', seed, flush=True)
            result_dir = OUT / f'seed{seed}'
            a1_checkpoint = checkpoint_dir / f'a1_seed{seed}.pt'
            v3_checkpoint = checkpoint_dir / f'v3_baseline_ema_seed{seed}.pt'
            subprocess.run([sys.executable, 'v3_a1_fusion.py', 'score-a1',
                            '--sources', str(source), '--checkpoint', str(a1_checkpoint),
                            '--device', 'cuda', '--output-dir', str(result_dir)], cwd=REPO, check=True)
            v3_file = result_dir / f'{source.stem}_v3.json'
            subprocess.run([sys.executable, 'evaluate_v3_quang.py', '_test', '--config', str(configs[seed]),
                            '--seed', str(seed), '--graph', str(graph),
                            '--expected-samples', str(len(truth)), '--signature', f'{source.stem}_ema_seed{seed}',
                            '--report', str(v3_file)], cwd=REPO, check=True)
            v3 = ev.read_json(v3_file)
            a1 = ev.read_json(result_dir / f'scores_{source.stem}.json')
            assert v3['checkpoint_sha256'] == ev.sha256(v3_checkpoint)
            assert a1['checkpoint_sha256'] == ev.sha256(a1_checkpoint)
            combined = paired_scores(source, truth, v3, a1)
            metrics = ev.score_probabilities(truth, combined, 0.5)
            result = dict(seed=seed, source=relative, source_sha256=source_digest(source),
                          samples=len(truth), threshold=0.5, metrics=metrics,
                          v3_sha256=ev.sha256(v3_checkpoint), a1_sha256=ev.sha256(a1_checkpoint),
                          y_true=truth.tolist(), p_human_fusion=combined.tolist())
            ev.write_json(result_dir / f'{source.stem}_fusion.json', result)
            for table, condition, noise in matching:
                all_results.append(dict(table=table, condition=condition, noise_percent=noise, seed=seed,
                                        source=relative, samples=len(truth), accuracy=metrics['accuracy'],
                                        f1=metrics['f1'], auc=metrics['auc']))
            pd.DataFrame(all_results).to_csv(OUT / 'results_by_seed.csv', index=False)
    finally:
        graph.unlink(missing_ok=True)
        print('?? x?a graph', graph.name, flush=True)

frame = pd.DataFrame(all_results)
summary = frame.groupby(['table', 'condition', 'noise_percent', 'source'], sort=False).agg(
    seeds=('seed', 'count'), accuracy_mean=('accuracy', 'mean'), accuracy_std=('accuracy', 'std'),
    accuracy_min=('accuracy', 'min'), accuracy_max=('accuracy', 'max')).reset_index()
summary.to_csv(OUT / 'summary.csv', index=False)
from IPython.display import display
for table in ('table2', 'table3', 'table4'):
    print(table.upper(), 'accuracy')
    display(frame[frame.table == table].pivot(index='condition', columns='seed', values='accuracy'))
    display(summary[summary.table == table])


In [ ]:
import zipfile
from IPython.display import FileLink, display
archive = WORK / 'v3_ema_a1_5seeds_table2_table3_table4.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as output:
    for path in OUT.rglob('*'):
        if path.is_file():
            output.write(path, path.relative_to(WORK))
    output.write(manifest_files[0], 'checkpoint_manifest.json')
print('K?t qu?:', archive)
display(FileLink(str(archive)))
